In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

- Calculate Agreement and Difference in Variance Explained (VE) between human and LLM respondents

In [9]:
def agreement_vediff(df):
    ve_by_group = df.groupby(df["group"].str.lower())["variance_explained"].mean()
    human_ve_mean = ve_by_group.get("human", float("nan"))
    llm_ve_mean   = ve_by_group.get("llm", float("nan"))
    ve_diff_llm_minus_human = llm_ve_mean - human_ve_mean
    consensus_by_group = df.groupby(df["group"].str.lower())["pct_agreement_consensus"].mean()
    cultural_consensus = consensus_by_group.get("llm", float("nan")) / 100
    return cultural_consensus, ve_diff_llm_minus_human

- Use the summaries generated by the R script (CCT_HLLM_Final.R) here

In [26]:
input_path = Path("ARR_MARCH") / "Output" / "summaries" / "CC_VE" 

In [3]:
single_culture = ['Japan', 'Armenia', 'Germany', 'Greece', 'Netherlands']

- Selecting only the columns that are required for Table 2

In [4]:
columns_needed = ['domain', 'country', 'group', 'variance_explained',
                  'comp_q99_mean', 'pct_agreement_consensus', 'GPT_OSS_120B',
                  'LLAMA3_1_70B', 'LLAMA3_70B', 'PHI3_INSTRUCT',
                  'QWEN2_5VL_32B', 'QWEN2_5VL_72B', 'QWEN2_5VL_7B',
                  'QWEN3_32B', 'QWEN_7B', 'GPT_4O']

In [5]:
table2_list = []

In [6]:
files = list(input_path.glob("*.csv"))
#files[:5]

In [10]:
for file in files: 
    dfread = pd.read_csv(file)
    for col in ["variance_explained", "comp_q99_mean"]:
        dfread[col] = pd.to_numeric(dfread[col], errors="coerce")
    df1 = dfread[columns_needed]
    dfsingle = df1[df1['country'].isin(single_culture)]
    dfmulti = df1[~df1['country'].isin(single_culture)]
    CC_single, VE_diff_single = agreement_vediff(dfsingle)
    CC_multi, VE_diff_multi = agreement_vediff(dfmulti)
    file_split = file.with_suffix('').name.split('_')
    if len(file_split) == 3:
        domain = file_split[-1]
    else:
        domain = file_split[2] + '_' + file_split[3]
    table2_list.append([domain, CC_multi, VE_diff_multi, CC_single, VE_diff_single])

In [11]:
table2_list[0]

['EV',
 np.float64(0.28),
 np.float64(0.13877880405262683),
 np.float64(0.32),
 np.float64(0.08207935194912722)]

In [12]:
dftable2 = pd.DataFrame(table2_list, columns = ['domain', 'cc_multi',
                                                'VE_Diff_multi', 'cc_single',
                                                'VE_Diff_single'])

In [13]:
num_cols = dftable2.select_dtypes(include="number").columns
dftable2[num_cols] = dftable2[num_cols].round(3)

In [14]:
#dftable2

In [15]:
df = dftable2.copy()

In [16]:
df["Domain"] = df["domain"].str.replace(r"_(I|II|III)$", "", regex=True)

In [17]:
dfagg = (df.groupby("Domain", as_index=False)[
            ["cc_multi", "VE_Diff_multi", "cc_single", "VE_Diff_single"]
        ].mean())

In [18]:
order = ["EV","EVN","HWB","PCPR","PIPP","POC","POM","POS","POST","RV","SCTOM","SVNS"]
dfagg["Domain"] = pd.Categorical(dfagg["Domain"], categories=order, ordered=True)
dfagg = dfagg.sort_values("Domain").set_index("Domain")

In [19]:
dfagg.columns = pd.MultiIndex.from_tuples([
    ("Multi-culture", "CC"),
    ("Multi-culture", "ΔVE"),
    ("Single-culture", "CC"),
    ("Single-culture", "ΔVE"),
])

- Domain becomes a column

In [20]:
paper_style = dfagg.reset_index()

In [21]:
paper_style.columns = pd.MultiIndex.from_tuples([
    ("", "Domain"),
    ("Multi-culture", "CC"),
    ("Multi-culture", "ΔVE"),
    ("Single-culture", "CC"),
    ("Single-culture", "ΔVE"),
])

In [22]:
paper_style

Multi-culture           Single-culture          
   Domain            CC       ΔVE             CC       ΔVE
0      EV      0.280000  0.139000       0.320000  0.082000
1     EVN      0.337000 -0.101000       0.495000 -0.216000
2     HWB      0.440000 -0.031000       0.360000 -0.305000
3    PCPR      0.238000  0.150500       0.486500  0.040000
4    PIPP      0.490000  0.071000       0.405000  0.034000
5     POC      0.800000  0.228000       0.880000  0.121000
6     POM      0.250000  0.121000       0.650000  0.085000
7     POS      0.771000  0.080000       0.371000 -0.071000
8    POST      0.360000  0.202000       0.480000  0.150000
9      RV      1.000000  0.167000       0.800000  0.176000
10  SCTOM      0.398000  0.045333       0.538667 -0.010667
11   SVNS      0.607667  0.042333       0.629667  0.065000

In [23]:
num_cols = paper_style.select_dtypes(include="number").columns

In [24]:
styled = (paper_style.style
          .format("{:.3f}", subset=num_cols)          
          .hide(axis="index")        
          .set_properties(**{"text-align": "center"})
          .set_properties(subset=[("", "Domain")], **{"text-align": "left"})
         )

In [25]:
display(styled)